### Checking different columns in files

In [1]:
#pyspark jupyter notebook matplotlib pandas pyarrow

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable  # Usa el Python del entorno actual
os.environ[("PYSPARK_DRI"
            "VER_PYTHON")] = sys.executable
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"  # Spark 4.2 requiere un Java compatible
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.python.worker.memory", "2g") \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.python.profile", "true") \
    .config("spark.python.connect.timeout", "60s") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/15 17:18:07 WARN Utils: Your hostname, DESKTOP-EKA7S1C, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/15 17:18:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/diego/repos/data-processing/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/15 17:18:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/15 17:18:08 WARN Utils: Service

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [4]:
import pyspark.pandas as ps
import pandas as pd
import re
import os
from pyspark.sql import Row
from pyspark.sql.types import *
from IPython.display import display, HTML
from pyspark.sql.functions import col, split, lit, count, when, trim, startswith, lower, to_date, datediff, avg, min, max, stddev, mean, kurtosis, skewness, lpad, expr, coalesce


/home/diego/repos/data-processing/.venv/lib/python3.12/site-packages/pyspark/pandas/__init__.py:33: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
!java -version

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-24.04-Ubuntu, mixed mode, sharing)


In [6]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # pretty print dataframe
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50) # pretty print dataframe
spark.sparkContext.setLogLevel("ERROR") # remove warnings
spark.conf.set('spark.rapids.sql.enabled','true')
spark.conf.set('spark.rapids.sql.incompatibleOps.enabled', 'true')
spark.conf.set('spark.rapids.sql.format.csv.read.enabled', 'true')
spark.conf.set('spark.rapids.sql.format.csv.enabled', 'true')

        

In [7]:
from pathlib import Path
from functools import reduce

data_dir = Path("data/processed/sivigila")
if not data_dir.is_dir():
    data_dir = Path("../data/processed/sivigila")

files = sorted(data_dir.glob("sivigila_weekly_municipal_*.parquet"))
if not files:
    raise FileNotFoundError(f"No yearly Parquet files found in {data_dir}")

sivigila_processed = reduce(
    lambda left, right: left.unionByName(right),
    (spark.read.parquet(str(path.resolve())) for path in files),
)

print(f"Loaded {len(files)} yearly files")

Loaded 19 yearly files


In [8]:
sivigila_processed.printSchema()

root
 |-- codigo_municipio: string (nullable = true)
 |-- nombre_municipio: string (nullable = true)
 |-- codigo_departamento: string (nullable = true)
 |-- ano_epidemiologico: long (nullable = true)
 |-- semana_epidemiologica: short (nullable = true)
 |-- inicio_semana: date (nullable = true)
 |-- fin_semana: date (nullable = true)
 |-- casos_total: long (nullable = true)
 |-- casos_masculinos: long (nullable = true)
 |-- casos_femeninos: long (nullable = true)
 |-- casos_hospitalizados: long (nullable = true)
 |-- casos_menor_1: long (nullable = true)
 |-- casos_1_4: long (nullable = true)
 |-- casos_5_14: long (nullable = true)
 |-- casos_15_49: long (nullable = true)
 |-- casos_50_mas: long (nullable = true)
 |-- casos_edad_desconocida: long (nullable = true)
 |-- casos_cabecera: long (nullable = true)
 |-- casos_centro_poblado: long (nullable = true)
 |-- casos_rural: byte (nullable = true)
 |-- casos_contributivo: long (nullable = true)
 |-- casos_subsidiado: long (nullable = tru

In [14]:
from pyspark.sql.functions import col, sum as spark_sum

sivigila_processed \
    .where(col("ano_epidemiologico") == 2023) \
    .agg(spark_sum("casos_total").alias("total_casos")) \
    .show()

+-----------+
|total_casos|
+-----------+
|      31469|
+-----------+



In [10]:

sivigila_processed.show(5, truncate=False)

+----------------+-------------------+-------------------+------------------+---------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+---------------------------

In [11]:
sivigila_processed.count()

226195

## Integración Datos Poblacion

In [12]:
from pyspark.sql import functions as F

csv_path = Path("data/processed/population/population_municipal_2007_2025.csv")
if not csv_path.is_file():
    csv_path = Path("../data/processed/population/population_municipal_2007_2025.csv")
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)

poblacion_csv = spark.read.option("header", True).csv(str(csv_path.resolve()))
poblacion_to_join = poblacion_csv.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico").cast("int").alias("ano_epidemiologico"),
    *(F.col(name).cast("long").alias(name)
      for name in poblacion_csv.columns
      if name not in {"codigo_municipio", "ano_epidemiologico"})
)
poblacion_to_join.printSchema()

root
 |-- codigo_municipio: string (nullable = true)
 |-- ano_epidemiologico: integer (nullable = true)
 |-- poblacion_total: long (nullable = true)
 |-- poblacion_hombres: long (nullable = true)
 |-- poblacion_mujeres: long (nullable = true)
 |-- poblacion_cabecera: long (nullable = true)
 |-- poblacion_rural: long (nullable = true)
 |-- poblacion_menor_1: long (nullable = true)
 |-- poblacion_1_4: long (nullable = true)
 |-- poblacion_5_14: long (nullable = true)
 |-- poblacion_15_49: long (nullable = true)
 |-- poblacion_50_mas: long (nullable = true)
 |-- poblacion_hombres_menor_1: long (nullable = true)
 |-- poblacion_hombres_1_4: long (nullable = true)
 |-- poblacion_hombres_5_14: long (nullable = true)
 |-- poblacion_hombres_15_49: long (nullable = true)
 |-- poblacion_hombres_50_mas: long (nullable = true)
 |-- poblacion_mujeres_menor_1: long (nullable = true)
 |-- poblacion_mujeres_1_4: long (nullable = true)
 |-- poblacion_mujeres_5_14: long (nullable = true)
 |-- poblacion_m

In [13]:
df_siv_pob = (
    sivigila_processed
    .join(
        poblacion_to_join,
        ["codigo_municipio", "ano_epidemiologico"],
        "left"
    )
)

In [14]:
df_siv_pob= (
    df_siv_pob

    # Total
    .withColumn(
        "incidencia_total_100k",
        F.try_divide(F.col("casos_total"), F.col("poblacion_total")) * 100000
    )

    # Área
    .withColumn(
        "incidencia_cabecera_100k",
        F.try_divide(F.col("casos_cabecera"), F.col("poblacion_cabecera")) * 100000
    )
    .withColumn(
        "incidencia_rural_100k",
        F.try_divide(
            F.col("casos_centro_poblado") + F.col("casos_rural"),
            F.col("poblacion_rural"),
        ) * 100000
    )

    # Edad
    .withColumn(
        "incidencia_menor_1_100k",
        F.try_divide(F.col("casos_menor_1"), F.col("poblacion_menor_1")) * 100000
    )
    .withColumn(
        "incidencia_1_4_100k",
        F.try_divide(F.col("casos_1_4"), F.col("poblacion_1_4")) * 100000
    )
    .withColumn(
        "incidencia_5_14_100k",
        F.try_divide(F.col("casos_5_14"), F.col("poblacion_5_14")) * 100000
    )
    .withColumn(
        "incidencia_15_49_100k",
        F.try_divide(F.col("casos_15_49"), F.col("poblacion_15_49")) * 100000
    )
    .withColumn(
        "incidencia_50_mas_100k",
        F.try_divide(F.col("casos_50_mas"), F.col("poblacion_50_mas")) * 100000
    )
    .withColumn(
        "incidencia_masculina_100k",
        F.try_divide(F.col("casos_masculinos"), F.col("poblacion_hombres")) * 100000
    )
    .withColumn(
        "incidencia_femenina_100k",
        F.try_divide(F.col("casos_femeninos"), F.col("poblacion_mujeres")) * 100000
    )
)

In [15]:
df_siv_pob.show()

+----------------+------------------+-------------------+-------------------+---------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+---------------------------

In [16]:
df_siv_pob.printSchema()

root
 |-- codigo_municipio: string (nullable = true)
 |-- ano_epidemiologico: long (nullable = true)
 |-- nombre_municipio: string (nullable = true)
 |-- codigo_departamento: string (nullable = true)
 |-- semana_epidemiologica: short (nullable = true)
 |-- inicio_semana: date (nullable = true)
 |-- fin_semana: date (nullable = true)
 |-- casos_total: long (nullable = true)
 |-- casos_masculinos: long (nullable = true)
 |-- casos_femeninos: long (nullable = true)
 |-- casos_hospitalizados: long (nullable = true)
 |-- casos_menor_1: long (nullable = true)
 |-- casos_1_4: long (nullable = true)
 |-- casos_5_14: long (nullable = true)
 |-- casos_15_49: long (nullable = true)
 |-- casos_50_mas: long (nullable = true)
 |-- casos_edad_desconocida: long (nullable = true)
 |-- casos_cabecera: long (nullable = true)
 |-- casos_centro_poblado: long (nullable = true)
 |-- casos_rural: byte (nullable = true)
 |-- casos_contributivo: long (nullable = true)
 |-- casos_subsidiado: long (nullable = tru

In [17]:
# 226195
df_siv_pob.count()

226195

## Integración cobertura vacunación

In [18]:
coverage_csv_path = Path("data/processed/vaccine_coverage/coberturas_municipales_2016_2025.csv")
if not coverage_csv_path.is_file():
    coverage_csv_path = Path("../data/processed/vaccine_coverage/coberturas_municipales_2016_2025.csv")
if not coverage_csv_path.is_file():
    raise FileNotFoundError(coverage_csv_path)

coberturas_vacunacion = (
    spark.read.option("header", True)
    .csv(str(coverage_csv_path.resolve()))
)
coberturas_vacunacion.printSchema()


root
 |-- CODEP: string (nullable = true)
 |-- Nombre_ET: string (nullable = true)
 |-- DIVIPOLA: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- meta_pob_1_anio: string (nullable = true)
 |-- varicela_un_anio_dosis_unica: string (nullable = true)
 |-- %_varicela_un_anio_dosis_unica: string (nullable = true)
 |-- meta_pob_5_anios: string (nullable = true)
 |-- varicela_cinco_anios_refuerzo: string (nullable = true)
 |-- %_varicela_cinco_anios_refuerzo: string (nullable = true)
 |-- anio: string (nullable = true)



In [19]:
columnas_cobertura = [
    name for name in coberturas_vacunacion.columns
    if name not in {"CODEP", "Nombre_ET", "DIVIPOLA", "Municipio", "anio"}
]

coberturas_para_join = coberturas_vacunacion.select(
    F.col("DIVIPOLA").alias("codigo_municipio"),
    F.col("anio").cast("long").alias("ano_epidemiologico"),
    *(F.col(name).cast("double").alias(name) for name in columnas_cobertura),
)

df_siv_pob_vac = df_siv_pob.join(
    coberturas_para_join,
    ["codigo_municipio", "ano_epidemiologico"],
    "left",
)


In [20]:
df_siv_pob_vac.show()

+----------------+------------------+-------------------+-------------------+---------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+---------------------------

In [21]:
# 226195
df_siv_pob_vac.count()

226195

In [22]:
df_siv_pob_vac.select("ano_epidemiologico").distinct().where(df_siv_pob_vac["meta_pob_1_anio"].isNull()).show()

+------------------+
|ano_epidemiologico|
+------------------+
|              2007|
|              2008|
|              2009|
|              2010|
|              2011|
|              2012|
|              2013|
|              2014|
|              2015|
+------------------+



In [23]:
df_siv_pob_vac.select("ano_epidemiologico").distinct().where(df_siv_pob_vac["meta_pob_1_anio"].isNotNull()).show()

+------------------+
|ano_epidemiologico|
+------------------+
|              2016|
|              2017|
|              2018|
|              2019|
|              2020|
|              2021|
|              2022|
|              2023|
|              2024|
|              2025|
+------------------+



## Precipitación CHIRPS sat

In [24]:
chirps_sat_dir = Path("data/processed/chirps_precipitation")
if not chirps_sat_dir.is_dir():
    chirps_sat_dir = Path("../data/processed/chirps_precipitation")

chirps_sat_files = [
    chirps_sat_dir / f"chirps_precipitation_weekly_municipal_sat_{anio}.parquet"
    for anio in range(2007, 2026)
]
archivos_faltantes = [path for path in chirps_sat_files if not path.is_file()]
if archivos_faltantes:
    raise FileNotFoundError(f"Faltan archivos CHIRPS sat: {archivos_faltantes}")

precipitacion_chirps_sat = spark.read.parquet(
    *[str(path.resolve()) for path in chirps_sat_files]
)
print(f"Cargados {len(chirps_sat_files)} archivos anuales CHIRPS sat")
precipitacion_chirps_sat.printSchema()

Cargados 19 archivos anuales CHIRPS sat
root
 |-- codigo_municipio: string (nullable = true)
 |-- nombre_municipio: string (nullable = true)
 |-- ano_epidemiologico: long (nullable = true)
 |-- semana_epidemiologica: long (nullable = true)
 |-- inicio_semana: string (nullable = true)
 |-- fin_semana: string (nullable = true)
 |-- precipitacion_total_mm: double (nullable = true)
 |-- fraccion_area_con_datos: double (nullable = true)
 |-- dias_con_datos: short (nullable = true)
 |-- dias_imputados: short (nullable = true)
 |-- variante_chirps: string (nullable = true)



In [25]:
precipitacion_chirps_sat.show(5, truncate=False)

+----------------+----------------+------------------+---------------------+-------------+----------+----------------------+-----------------------+--------------+--------------+---------------+
|codigo_municipio|nombre_municipio|ano_epidemiologico|semana_epidemiologica|inicio_semana|fin_semana|precipitacion_total_mm|fraccion_area_con_datos|dias_con_datos|dias_imputados|variante_chirps|
+----------------+----------------+------------------+---------------------+-------------+----------+----------------------+-----------------------+--------------+--------------+---------------+
|05001           |MEDELLÍN        |2025              |1                    |2024-12-29   |2025-01-04|45.09285281069118     |1.0                    |7             |0             |sat            |
|05002           |ABEJORRAL       |2025              |1                    |2024-12-29   |2025-01-04|44.529509746474346    |1.0                    |7             |0             |sat            |
|05004           |ABRIAQU

In [26]:
chirps_sat_para_join = precipitacion_chirps_sat.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico"),
    F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
    F.col("precipitacion_total_mm").alias("precipitacion_total_mm_chirps_sat"),
    F.col("fraccion_area_con_datos").alias("fraccion_area_con_datos_chirps_sat"),
    F.col("dias_con_datos").alias("dias_con_datos_chirps_sat"),
    F.col("dias_imputados").alias("dias_imputados_chirps_sat"),
)

df_siv_pob_vac_chirps_sat = df_siv_pob_vac.join(
    chirps_sat_para_join,
    ["codigo_municipio", "ano_epidemiologico", "semana_epidemiologica"],
    "left",
)

In [27]:
df_siv_pob_vac_chirps_sat.count()

226195

In [28]:
df_siv_pob_vac_chirps_sat.where(col("precipitacion_total_mm_chirps_sat").isNull()).count()

0

In [29]:
df_siv_pob_vac_chirps_sat.select(
    "codigo_municipio", "ano_epidemiologico", "semana_epidemiologica",
    "precipitacion_total_mm_chirps_sat", "fraccion_area_con_datos_chirps_sat",
).show(5, truncate=False)

+----------------+------------------+---------------------+---------------------------------+----------------------------------+
|codigo_municipio|ano_epidemiologico|semana_epidemiologica|precipitacion_total_mm_chirps_sat|fraccion_area_con_datos_chirps_sat|
+----------------+------------------+---------------------+---------------------------------+----------------------------------+
|11001           |2007              |1                    |1.946187169371888                |1.0                               |
|13001           |2007              |1                    |0.06965742833352949              |0.9999999999999999                |
|25754           |2007              |1                    |0.9843904358946847               |0.9999999999999998                |
|41206           |2007              |1                    |3.9904635969366176               |1.0                               |
|54001           |2007              |1                    |5.617771868934921                |0.99

## Humedad ERA5-Land

In [30]:
humidity_dir = Path("data/processed/climate/humidity")
if not humidity_dir.is_dir():
    humidity_dir = Path("../data/processed/climate/humidity")

humidity_files = [
    humidity_dir / f"humidity_weekly_municipal_{anio}.parquet"
    for anio in range(2007, 2026)
]
humidity_missing = [path for path in humidity_files if not path.is_file()]
if humidity_missing:
    raise FileNotFoundError(f"Faltan archivos de humedad: {humidity_missing}")

humedad_era5_land = spark.read.parquet(
    *[str(path.resolve()) for path in humidity_files]
)
print(f"Cargados {len(humidity_files)} archivos anuales de humedad")
humedad_era5_land.printSchema()

Cargados 19 archivos anuales de humedad
root
 |-- codigo_municipio: string (nullable = true)
 |-- nombre_municipio: string (nullable = true)
 |-- ano_epidemiologico: long (nullable = true)
 |-- semana_epidemiologica: long (nullable = true)
 |-- inicio_semana: string (nullable = true)
 |-- fin_semana: string (nullable = true)
 |-- humedad_relativa_media_pct: double (nullable = true)
 |-- fraccion_area_con_datos_humedad_relativa_media_pct: double (nullable = true)
 |-- humedad_especifica_media_g_kg: double (nullable = true)
 |-- fraccion_area_con_datos_humedad_especifica_media_g_kg: double (nullable = true)
 |-- humedad_absoluta_media_g_m3: double (nullable = true)
 |-- fraccion_area_con_datos_humedad_absoluta_media_g_m3: double (nullable = true)



In [31]:
columnas_humedad = [
    "humedad_relativa_media_pct",
    "fraccion_area_con_datos_humedad_relativa_media_pct",
    "humedad_especifica_media_g_kg",
    "fraccion_area_con_datos_humedad_especifica_media_g_kg",
    "humedad_absoluta_media_g_m3",
    "fraccion_area_con_datos_humedad_absoluta_media_g_m3",
]
humedad_para_join = humedad_era5_land.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico"),
    F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
    *(F.col(name) for name in columnas_humedad),
)

df_siv_pob_vac_chirps_sat_humedad = df_siv_pob_vac_chirps_sat.join(
    humedad_para_join,
    ["codigo_municipio", "ano_epidemiologico", "semana_epidemiologica"],
    "left",
)

In [32]:
# las de providencia que tuvo que extraerse por hora dado que por día no fue posibe tener datos de providencia
df_siv_pob_vac_chirps_sat_humedad.where(col("humedad_relativa_media_pct").isNull()).count()

57

In [33]:
df_siv_pob_vac_chirps_sat_humedad.select("ano_epidemiologico").where(col("humedad_relativa_media_pct").isNull()).distinct().sort("ano_epidemiologico").show()


+------------------+
|ano_epidemiologico|
+------------------+
|              2009|
|              2012|
|              2013|
|              2014|
|              2016|
|              2017|
|              2018|
|              2019|
|              2020|
|              2021|
|              2022|
|              2023|
|              2024|
|              2025|
+------------------+



In [34]:
df_siv_pob_vac_chirps_sat_humedad.count()

226195

In [35]:
df_siv_pob_vac_chirps_sat_humedad.select(
    "codigo_municipio", "ano_epidemiologico", "semana_epidemiologica",
    "humedad_relativa_media_pct", "humedad_especifica_media_g_kg",
    "humedad_absoluta_media_g_m3",
).show(5, truncate=False)

+----------------+------------------+---------------------+--------------------------+-----------------------------+---------------------------+
|codigo_municipio|ano_epidemiologico|semana_epidemiologica|humedad_relativa_media_pct|humedad_especifica_media_g_kg|humedad_absoluta_media_g_m3|
+----------------+------------------+---------------------+--------------------------+-----------------------------+---------------------------+
|05045           |2015              |1                    |81.15478332107087         |17.53079385647656            |19.88838099627974          |
|05088           |2008              |1                    |81.85199153320386         |11.391704027751192           |10.757435930963252         |
|05088           |2019              |1                    |61.78807304641915         |9.201890190107553            |8.664469363889584          |
|05101           |2008              |1                    |86.65123363814406         |13.250225228620389           |13.20184593928

## Completar humedad de Providencia

ERA5-Land no tiene humedad válida para Providencia. Se usan las estimaciones horarias de ERA5 atmosférico solo en sus semanas faltantes y se conserva la fuente de cada fila.

In [36]:
providencia_humidity_dir = Path("data/processed/climate/providencia/humidity")
if not providencia_humidity_dir.is_dir():
    providencia_humidity_dir = Path("../data/processed/climate/providencia/humidity")

providencia_humidity_files = [
    providencia_humidity_dir / f"humidity_weekly_providencia_{anio}.parquet"
    for anio in range(2007, 2026)
]
providencia_missing = [path for path in providencia_humidity_files if not path.is_file()]
if providencia_missing:
    raise FileNotFoundError(f"Faltan archivos de humedad de Providencia: {providencia_missing}")

humedad_providencia = spark.read.parquet(
    *[str(path.resolve()) for path in providencia_humidity_files]
)
humedad_providencia_para_join = humedad_providencia.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico"),
    F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
    *(F.col(name).alias(f"{name}_providencia") for name in columnas_humedad),
    F.col("fuente_dato").alias("fuente_humedad_providencia"),
    F.col("es_estimacion").alias("es_estimacion_providencia"),
)

humedad_con_providencia = df_siv_pob_vac_chirps_sat_humedad.join(
    humedad_providencia_para_join,
    ["codigo_municipio", "ano_epidemiologico", "semana_epidemiologica"],
    "left",
).withColumn(
    "_usar_humedad_providencia",
    (F.col("codigo_municipio") == "88564")
    & F.col("humedad_relativa_media_pct").isNull()
    & F.col("humedad_relativa_media_pct_providencia").isNotNull(),
)

fuente_previa = (
    F.col("fuente_humedad") if "fuente_humedad" in humedad_con_providencia.columns
    else F.lit("ERA5-Land_daily_UTC")
)
estimacion_previa = (
    F.col("es_estimacion_humedad") if "es_estimacion_humedad" in humedad_con_providencia.columns
    else F.lit(False)
)

for name in columnas_humedad:
    humedad_con_providencia = humedad_con_providencia.withColumn(
        name,
        F.when(F.col("_usar_humedad_providencia"), F.col(f"{name}_providencia"))
        .otherwise(F.col(name)),
    )

df_siv_pob_vac_chirps_sat_humedad = (
    humedad_con_providencia
    .withColumn(
        "fuente_humedad",
        F.when(F.col("_usar_humedad_providencia"), F.col("fuente_humedad_providencia"))
        .otherwise(fuente_previa),
    )
    .withColumn(
        "es_estimacion_humedad",
        F.when(F.col("_usar_humedad_providencia"), F.col("es_estimacion_providencia"))
        .otherwise(estimacion_previa),
    )
    .drop(
        "_usar_humedad_providencia",
        "fuente_humedad_providencia",
        "es_estimacion_providencia",
        *(f"{name}_providencia" for name in columnas_humedad),
    )
)

In [37]:
df_siv_pob_vac_chirps_sat_humedad.groupBy("fuente_humedad", "es_estimacion_humedad").agg(
    F.count("*").alias("filas"),
    F.sum(F.col("humedad_relativa_media_pct").isNull().cast("int")).alias("humedad_relativa_nula"),
).show(truncate=False)

+--------------------------+---------------------+------+---------------------+
|fuente_humedad            |es_estimacion_humedad|filas |humedad_relativa_nula|
+--------------------------+---------------------+------+---------------------+
|ERA5-Land_daily_UTC       |false                |226138|0                    |
|ERA5_atmosphere_hourly_EDH|true                 |57    |0                    |
+--------------------------+---------------------+------+---------------------+



In [38]:
df_siv_pob_vac_chirps_sat_humedad.show()

+----------------+------------------+---------------------+----------------+-------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+------------------------------

## Precipitación ERA5-Land y Providencia

Se completa la precipitación nula de Providencia (`88564`) con la estimación horaria de ERA5 atmosférico. La cobertura y la fuente corresponden al valor seleccionado.

In [39]:
precipitation_dir = Path("data/processed/climate/precipitation")
if not precipitation_dir.is_dir():
    precipitation_dir = Path("../data/processed/climate/precipitation")

providencia_precipitation_dir = Path("data/processed/climate/providencia/precipitation")
if not providencia_precipitation_dir.is_dir():
    providencia_precipitation_dir = Path("../data/processed/climate/providencia/precipitation")

precipitation_files = [
    precipitation_dir / f"precipitation_weekly_municipal_{anio}.parquet"
    for anio in range(2007, 2026)
]
providencia_precipitation_files = [
    providencia_precipitation_dir / f"precipitation_weekly_providencia_{anio}.parquet"
    for anio in range(2007, 2026)
]
missing_precipitation = [
    path for path in precipitation_files + providencia_precipitation_files
    if not path.is_file()
]
if missing_precipitation:
    raise FileNotFoundError(f"Faltan archivos de precipitación: {missing_precipitation}")

precipitacion_era5_land = spark.read.parquet(
    *[str(path.resolve()) for path in precipitation_files]
)
precipitacion_providencia = spark.read.parquet(
    *[str(path.resolve()) for path in providencia_precipitation_files]
)

llaves_clima = ["codigo_municipio", "ano_epidemiologico", "semana_epidemiologica"]
precipitacion_providencia_para_join = precipitacion_providencia.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico"),
    F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
    F.col("precipitacion_total_mm").alias("precipitacion_total_mm_providencia"),
    F.col("fraccion_area_con_datos").alias("fraccion_area_con_datos_providencia"),
    F.col("fuente_dato").alias("fuente_precipitacion_providencia"),
    F.col("es_estimacion").alias("es_estimacion_precipitacion_providencia"),
)

precipitacion_completa = (
    precipitacion_era5_land.select(
        F.col("codigo_municipio"),
        F.col("ano_epidemiologico"),
        F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
        F.col("precipitacion_total_mm"),
        F.col("fraccion_area_con_datos"),
    )
    .join(precipitacion_providencia_para_join, llaves_clima, "left")
    .withColumn(
        "_usar_precipitacion_providencia",
        (F.col("codigo_municipio") == "88564")
        & F.col("precipitacion_total_mm").isNull()
        & F.col("precipitacion_total_mm_providencia").isNotNull(),
    )
)

precipitacion_para_join = precipitacion_completa.select(
    *llaves_clima,
    F.when(
        F.col("_usar_precipitacion_providencia"),
        F.col("precipitacion_total_mm_providencia"),
    ).otherwise(F.col("precipitacion_total_mm")).alias("precipitacion_total_mm_era5"),
    F.when(
        F.col("_usar_precipitacion_providencia"),
        F.col("fraccion_area_con_datos_providencia"),
    ).otherwise(F.col("fraccion_area_con_datos")).alias("fraccion_area_con_datos_precipitacion_era5"),
    F.when(
        F.col("_usar_precipitacion_providencia"),
        F.col("fuente_precipitacion_providencia"),
    ).when(
        F.col("precipitacion_total_mm").isNotNull(), F.lit("ERA5-Land_daily_UTC")
    ).alias("fuente_precipitacion"),
    F.when(
        F.col("_usar_precipitacion_providencia"),
        F.col("es_estimacion_precipitacion_providencia"),
    ).when(
        F.col("precipitacion_total_mm").isNotNull(), F.lit(False)
    ).alias("es_estimacion_precipitacion"),
)

df_siv_pob_vac_chirps_sat_humedad_precipitacion = (
    df_siv_pob_vac_chirps_sat_humedad
    .join(precipitacion_para_join, llaves_clima, "left")
)

In [41]:
df_siv_pob_vac_chirps_sat_humedad_precipitacion.groupBy(
    "fuente_precipitacion", "es_estimacion_precipitacion"
).agg(
    F.count("*").alias("filas"),
    F.sum(F.col("precipitacion_total_mm_era5").isNull().cast("int")).alias("precipitacion_nula"),
).show(truncate=False)



+--------------------------+---------------------------+------+------------------+
|fuente_precipitacion      |es_estimacion_precipitacion|filas |precipitacion_nula|
+--------------------------+---------------------------+------+------------------+
|ERA5-Land_daily_UTC       |false                      |226138|0                 |
|ERA5_atmosphere_hourly_EDH|true                       |57    |0                 |
+--------------------------+---------------------------+------+------------------+



In [42]:
df_siv_pob_vac_chirps_sat_humedad_precipitacion.where(
    F.col("codigo_municipio") == "88564"
).select(
    "codigo_municipio", "nombre_municipio", "ano_epidemiologico", "semana_epidemiologica",
    "precipitacion_total_mm_era5", "fraccion_area_con_datos_precipitacion_era5",
    "fuente_precipitacion", "es_estimacion_precipitacion",
).show(5, truncate=False)

+----------------+----------------+------------------+---------------------+---------------------------+------------------------------------------+--------------------------+---------------------------+
|codigo_municipio|nombre_municipio|ano_epidemiologico|semana_epidemiologica|precipitacion_total_mm_era5|fraccion_area_con_datos_precipitacion_era5|fuente_precipitacion      |es_estimacion_precipitacion|
+----------------+----------------+------------------+---------------------+---------------------------+------------------------------------------+--------------------------+---------------------------+
|88564           |PROVIDENCIA     |2009              |19                   |6.636002165499351          |1.0                                       |ERA5_atmosphere_hourly_EDH|true                       |
|88564           |PROVIDENCIA     |2009              |17                   |11.185043332678061         |1.0                                       |ERA5_atmosphere_hourly_EDH|true          

## Temperatura ERA5-Land y Providencia

Se completan únicamente las semanas sin temperatura de Providencia (`88564`) con la estimación horaria de ERA5 atmosférico. Se conserva la cobertura y la procedencia del valor utilizado.

In [43]:
temperature_dir = Path("data/processed/climate/temperature")
if not temperature_dir.is_dir():
    temperature_dir = Path("../data/processed/climate/temperature")

providencia_temperature_dir = Path("data/processed/climate/providencia/temperature")
if not providencia_temperature_dir.is_dir():
    providencia_temperature_dir = Path("../data/processed/climate/providencia/temperature")

temperature_files = [
    temperature_dir / f"temperature_weekly_municipal_{anio}.parquet"
    for anio in range(2007, 2026)
]
providencia_temperature_files = [
    providencia_temperature_dir / f"temperature_weekly_providencia_{anio}.parquet"
    for anio in range(2007, 2026)
]
missing_temperature = [
    path for path in temperature_files + providencia_temperature_files
    if not path.is_file()
]
if missing_temperature:
    raise FileNotFoundError(f"Faltan archivos de temperatura: {missing_temperature}")

temperatura_era5_land = spark.read.parquet(
    *[str(path.resolve()) for path in temperature_files]
)
temperatura_providencia = spark.read.parquet(
    *[str(path.resolve()) for path in providencia_temperature_files]
)

temperatura_providencia_para_join = temperatura_providencia.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico"),
    F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
    F.col("temperatura_media_c").alias("temperatura_media_c_providencia"),
    F.col("fraccion_area_con_datos").alias("fraccion_area_con_datos_temperatura_providencia"),
    F.col("fuente_dato").alias("fuente_temperatura_providencia"),
    F.col("es_estimacion").alias("es_estimacion_temperatura_providencia"),
)

temperatura_completa = (
    temperatura_era5_land.select(
        F.col("codigo_municipio"),
        F.col("ano_epidemiologico"),
        F.col("semana_epidemiologica").cast("short").alias("semana_epidemiologica"),
        F.col("temperatura_media_c"),
        F.col("fraccion_area_con_datos"),
    )
    .join(temperatura_providencia_para_join, llaves_clima, "left")
    .withColumn(
        "_usar_temperatura_providencia",
        (F.col("codigo_municipio") == "88564")
        & F.col("temperatura_media_c").isNull()
        & F.col("temperatura_media_c_providencia").isNotNull(),
    )
)

temperatura_para_join = temperatura_completa.select(
    *llaves_clima,
    F.when(
        F.col("_usar_temperatura_providencia"),
        F.col("temperatura_media_c_providencia"),
    ).otherwise(F.col("temperatura_media_c")).alias("temperatura_media_c_era5"),
    F.when(
        F.col("_usar_temperatura_providencia"),
        F.col("fraccion_area_con_datos_temperatura_providencia"),
    ).otherwise(F.col("fraccion_area_con_datos")).alias("fraccion_area_con_datos_temperatura_era5"),
    F.when(
        F.col("_usar_temperatura_providencia"),
        F.col("fuente_temperatura_providencia"),
    ).when(
        F.col("temperatura_media_c").isNotNull(), F.lit("ERA5-Land_daily_UTC")
    ).alias("fuente_temperatura"),
    F.when(
        F.col("_usar_temperatura_providencia"),
        F.col("es_estimacion_temperatura_providencia"),
    ).when(
        F.col("temperatura_media_c").isNotNull(), F.lit(False)
    ).alias("es_estimacion_temperatura"),
)

df_siv_pob_vac_chirps_sat_humedad_precipitacion_temperatura = (
    df_siv_pob_vac_chirps_sat_humedad_precipitacion
    .join(temperatura_para_join, llaves_clima, "left")
)

In [44]:
df_siv_pob_vac_chirps_sat_humedad_precipitacion_temperatura.groupBy(
    "fuente_temperatura", "es_estimacion_temperatura"
).agg(
    F.count("*").alias("filas"),
    F.sum(F.col("temperatura_media_c_era5").isNull().cast("int")).alias("temperatura_nula"),
).show(truncate=False)

+--------------------------+-------------------------+------+----------------+
|fuente_temperatura        |es_estimacion_temperatura|filas |temperatura_nula|
+--------------------------+-------------------------+------+----------------+
|ERA5-Land_daily_UTC       |false                    |226138|0               |
|ERA5_atmosphere_hourly_EDH|true                     |57    |0               |
+--------------------------+-------------------------+------+----------------+



In [45]:


df_siv_pob_vac_chirps_sat_humedad_precipitacion_temperatura.where(
    F.col("codigo_municipio") == "88564"
).select(
    "codigo_municipio", "nombre_municipio", "ano_epidemiologico", "semana_epidemiologica",
    "temperatura_media_c_era5", "fraccion_area_con_datos_temperatura_era5",
    "fuente_temperatura", "es_estimacion_temperatura",
).show(5, truncate=False)

+----------------+----------------+------------------+---------------------+------------------------+----------------------------------------+--------------------------+-------------------------+
|codigo_municipio|nombre_municipio|ano_epidemiologico|semana_epidemiologica|temperatura_media_c_era5|fraccion_area_con_datos_temperatura_era5|fuente_temperatura        |es_estimacion_temperatura|
+----------------+----------------+------------------+---------------------+------------------------+----------------------------------------+--------------------------+-------------------------+
|88564           |PROVIDENCIA     |2009              |19                   |26.824686604327354      |1.0                                     |ERA5_atmosphere_hourly_EDH|true                     |
|88564           |PROVIDENCIA     |2009              |17                   |26.348251137091193      |1.0                                     |ERA5_atmosphere_hourly_EDH|true                     |
|88564           |PR

## Guardar dataset integrado

El resultado se guarda como un dataset Parquet de Spark, con todos los municipios y semanas disponibles.

In [ ]:
integrated_data_dir = Path("data/processed/integration_data")
if not integrated_data_dir.parent.is_dir():
    integrated_data_dir = Path("../data/processed/integration_data")
integrated_data_dir.mkdir(parents=True, exist_ok=True)

integrated_data_path = integrated_data_dir / "sivigila_integrado_semanal_municipal.parquet"
(
    df_siv_pob_vac_chirps_sat_humedad_precipitacion_temperatura
    .coalesce(4)
    .write
    .mode("overwrite")
    .parquet(str(integrated_data_path.resolve()))
)
print(f"Dataset integrado guardado en {integrated_data_path.resolve()}")